# Solution 5 — CSTR simulation checkpoint

The full estimation and control formulation is worked in chapters 14–15; this solution verifies the reusable simulation boundary.


In [ ]:
# Locate the repository so these editable modules work without installation.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Reload edited model modules before each cell during tutorial development.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from casadi_tutorial.models.cstr import CSTRParameters, CSTRSystem

parameters = CSTRParameters()
system = CSTRSystem(parameters)

# Check the documented operating point before running a simulation.
residual = system.steady_state_residual()
assert np.linalg.norm(residual, np.inf) < 2e-4

step_size = 0.02
number_of_steps = 25
step = system.discrete_dynamics(step_size)
x_history = [parameters.nominal_state]
u_history = np.full(
    number_of_steps,
    parameters.nominal_jacket_temperature + 5,
)
for u_value in u_history:
    x_next = step(
        x_history[-1],
        [u_value],
        parameters.nominal_disturbance,
    )
    x_history.append(np.asarray(x_next.full()).ravel())

x_history = np.asarray(x_history)
assert x_history.shape == (number_of_steps + 1, 2)
assert np.isfinite(x_history).all()

# Plot both state components and the held manipulated input over time.
x_time = step_size * np.arange(number_of_steps + 1)
u_time = x_time[:-1]
figure, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 7))
axes[0].plot(x_time, x_history[:, 0])
axes[0].set_ylabel("x[0] concentration")
axes[1].plot(x_time, x_history[:, 1])
axes[1].set_ylabel("x[1] temperature")
axes[2].step(u_time, u_history, where="post")
axes[2].set(xlabel="time [h]", ylabel="u jacket [K]")
for axis in axes:
    axis.grid(True)
figure.tight_layout()
plt.show()